<h1> GAIA - TOOL CALLING 

In [1]:
!pip install plotly
!pip install --upgrade nbformat

In [2]:
from astroquery.gaia import Gaia
import matplotlib.pyplot as plt 
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML
from astroquery.gaia import Gaia
import astropy.table
import nbformat
print(nbformat.__version__)

5.10.4


In [2]:
query = f"SELECT source_id, ra, dec, pmra, pmdec, parallax \
            FROM gaiadr3.gaia_source \
            WHERE has_epoch_photometry = 'True' \
            AND has_xp_sampled = 'True'\
            AND has_rvs = 'True' \
            AND has_mcmc_msc = 'True' \
            AND has_mcmc_gspphot = 'True' \
            AND random_index between 0 and 200000"

adql_query = """
SELECT TOP 100 
source_id, ra, dec, parallax, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE parallax IS NOT NULL 
ORDER BY phot_g_mean_mag ASC
"""

adql_query = """
SELECT TOP 100 
source_id, ra, dec, parallax, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE phot_g_mean_mag < 3
AND parallax IS NOT NULL
"""

adql_query = """SELECT
source_id, ra, dec, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE phot_g_mean_mag < 6 
AND parallax IS NOT NULL 
ORDER BY phot_g_mean_mag ASC
"""

job = Gaia.launch_job_async(adql_query)

results = job.get_results()
print(f'Table size (rows): {len(results)}')
results     

#We need to run jobs asynchronously, otherwise long jobs exhaust the compute nodes in the search engine.

500 Error 500:
null


HTTPError: Error 500:
null

In [3]:
# The SQL (ADQL) equivalent
adql_query = """
SELECT 
    source_id, ra, dec, parallax, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 56.75, 24.116, 0.5)
)
"""

job = Gaia.launch_job_async(adql_query)
results = job.get_results()
#job.get_data()
#job.get_results()
job.is_finished()

INFO: Query finished. [astroquery.utils.tap.core]


True

In [4]:
def get_gaia_cone_search(ra, dec, radius_deg=0.5, table="gaiadr3.gaia_source", verbose = False):
    """
    Performs a robust Gaia ADQL cone search with input validation and error handling.
    """
    
    if not all(isinstance(x, (int, float)) for x in [ra, dec, radius_deg]):
        raise ValueError("RA, Dec, and Radius must be numeric (int or float).")
    
    if not (0 <= ra <= 360):
        raise ValueError(f"RA must be between 0 and 360 degrees. Got: {ra}")
    
    if not (-90 <= dec <= 90):
        raise ValueError(f"Dec must be between -90 and 90 degrees. Got: {dec}")
        
    if radius_deg <= 0 or radius_deg > 5:
        raise ValueError(f"Radius must be > 0 and <= 5 degrees. Got: {radius_deg}")

    adql_query = f"""
    SELECT 
        source_id, ra, dec, parallax, phot_g_mean_mag 
    FROM {table}
    WHERE 1 = CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius_deg})
    )
    """
        
    try:
        if verbose:
            print(f"Querying {table}...")
    
        job = Gaia.launch_job_async(adql_query)
        results = job.get_results()
        
        if results is None or len(results) == 0:
            if verbose: 
                print("Warning: Query returned 0 results. Check your coordinates.")
            return astropy.table.Table() # Return empty table instead of crashing
            
        return results

    except Exception as e:
        print(f"An error occurred during the Gaia query: {e}")
        raise

In [5]:
df = results.to_pandas().dropna(subset=['ra', 'dec'])
ra_rad = np.radians(df['ra'].values)
dec_rad = np.radians(df['dec'].values)

x_stars = np.cos(dec_rad) * np.cos(ra_rad)
y_stars = np.cos(dec_rad) * np.sin(ra_rad)
z_stars = np.sin(dec_rad)

fig = go.Figure()

for r in np.arange(0, 360, 30): 
    r_rad = np.radians(r)
    d_range = np.linspace(-np.pi/2, np.pi/2, 50)
    x = np.cos(d_range) * np.cos(r_rad)
    y = np.cos(d_range) * np.sin(r_rad)
    z = np.sin(d_range)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', 
                               line=dict(color='rgba(100,100,100,0.3)', width=1), 
                               hoverinfo='skip'))

for d in np.arange(-60, 90, 30): 
    d_rad = np.radians(d)
    r_range = np.linspace(0, 2*np.pi, 100)
    x = np.cos(d_rad) * np.cos(r_range)
    y = np.cos(d_rad) * np.sin(r_range)
    z = np.full_like(x, np.sin(d_rad))
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', 
                               line=dict(color='rgba(100,100,100,0.3)', width=1), 
                               hoverinfo='skip'))

fig.add_trace(go.Scatter3d(
    x=x_stars, y=y_stars, z=z_stars,
    mode='markers',
    marker=dict(size=2, color='white', opacity=0.9),
    name='Stars'
))

fig.update_layout(
    template="plotly_dark",
    scene=dict(
        xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
        bgcolor="black",
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    showlegend=False
)

plot_html = fig.to_html(full_html=False, include_plotlyjs='cdn')
display(HTML(plot_html))

In [9]:
def get_gaia_color_analysis(ra, dec, radius_deg=0.5):
    """
    Robust tool to extract star colors (BP-RP) for a specific sky region.
    Validates inputs and filters for high-quality photometric data.
    """
    # 1. Input Guardrails
    if not (0 <= ra <= 360) or not (-90 <= dec <= 90):
        return {"error": "Invalid coordinates. RA must be [0,360], Dec [-90,90]."}
    
    if radius_deg <= 0 or radius_deg > 2.0:
        return {"error": "Radius must be between 0 and 2 degrees for stability."}

    # 2. ADQL Query with built-in quality filtering
    # We filter out NULLs to avoid NaNs in the histogram
    adql_query = f"""
    SELECT 
        source_id, ra, dec, phot_g_mean_mag, 
        phot_bp_mean_mag, phot_rp_mean_mag,
        (phot_bp_mean_mag - phot_rp_mean_mag) AS bp_rp
    FROM gaiadr3.gaia_source 
    WHERE 1 = CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius_deg})
    )
    AND phot_bp_mean_mag IS NOT NULL 
    AND phot_rp_mean_mag IS NOT NULL
    AND phot_g_mean_mag < 19  -- Filter for stars bright enough for reliable color
    """

    try:
        job = Gaia.launch_job_async(adql_query)
        results = job.get_results()
        
        if len(results) == 0:
            return {"status": "empty", "message": "No stars with valid color data found here."}
            
        return results.to_pandas()

    except Exception as e:
        return {"status": "error", "message": str(e)}

In [10]:
import plotly.express as px
from IPython.display import HTML

def plot_stellar_colors(df):
    if isinstance(df, dict): # Handle the error/empty cases from the tool
        print(f"Cannot plot: {df.get('message', 'Unknown error')}")
        return

    # Create the histogram
    fig = px.histogram(
        df, x="bp_rp", 
        nbins=100,
        title="Celestial Color Profile (BP - RP Index)",
        color_discrete_sequence=['#636EFA'],
        template="plotly_dark"
    )

    fig.update_layout(
        xaxis_title="Color Index (Blue ← 0 → Red)",
        yaxis_title="Star Count",
        bargap=0.05
    )
    
    # Overlay temperature zones
    fig.add_vrect(x0=-0.5, x1=0.5, fillcolor="blue", opacity=0.1, annotation_text="Hot/Blue")
    fig.add_vrect(x0=1.5, x1=4.0, fillcolor="red", opacity=0.1, annotation_text="Cool/Red")

    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

# Run it
data = get_gaia_color_analysis(56.75, 24.116, 0.5)

print(data)

plot_stellar_colors(data)

INFO: Query finished. [astroquery.utils.tap.core]
              source_id         ra        dec  phot_g_mean_mag  \
0     65008938527293824  56.635706  23.639561        17.125151   
1     65010274260706176  56.826320  23.691795        18.228733   
2     65010381636296320  56.819837  23.715562        16.082758   
3     65010514778903936  56.835506  23.735428        18.411253   
4     65011240629757568  56.736299  23.733105        17.285870   
...                 ...        ...        ...              ...   
3019  66786913252700800  56.584729  24.515465        18.938622   
3020  66788876054524160  56.798307  24.490412        17.279747   
3021  66789696390663808  56.821021  24.557399        18.229776   
3022  66797981385051904  56.439746  24.519736        16.253550   
3023  66798221903217280  56.496492  24.526892        14.598133   

      phot_bp_mean_mag  phot_rp_mean_mag     bp_rp  
0            18.557047         15.912810  2.644237  
1            18.760048         17.575434  1.184614 